# Part 3 — Deep Learning Methods

In this notebook we demonstrate **multi-omic integration using neural network encoders**.

The data are assumed to already be prepared:

- `../../data_tmp/TCGA-BRCA/transcriptomics.pkl`
- `../../data_tmp/TCGA-BRCA/proteomics.pkl`
- `../../data_tmp/TCGA-BRCA/methylation.pkl`

Each table is already sample-aligned, has no missing patient data, and uses the patient ID as the index.  
The prediction target is the `subtype` column.

## Workshop goals

1. Understand what a neural network is in this setting.
2. Build a simple early-integration neural network.
3. Build a multi-modal encoder with one encoder per omic view.
4. Compare both approaches using the same train/test split.
5. Extract a learned embedding space that can be used as a multi-omic patient profile.

## Key takeaways

- Deep learning methods can combine information shared across omics with information that is specific to individual omics.
- Supervised neural networks learn representations that are useful for the prediction task.
- Multi-modal encoder embeddings can be reused as compact multi-omic profiles.

## Outcome

By the end of this notebook, you will have a trained neural network model that produces patient-level multi-omic embeddings.  
These embeddings point naturally toward the next part of the workshop: using learned representations for downstream biological interpretation, clustering, or patient stratification.


In [ ]:
# Core imports

from pathlib import Path
import random

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

RANDOM_STATE = 42

def set_seed(seed: int = RANDOM_STATE):
    """Make the notebook results reasonably reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 1. Load the already prepared omics data

No filtering, alignment, imputation, or patient-level processing is needed here.

We only separate:

- `subtype` as the target
- all remaining columns as omic features
- the index as the patient ID


In [ ]:
DATA_DIR = Path("../../data_tmp/TCGA-BRCA")

omics = {
    "transcriptomics": pd.read_pickle(DATA_DIR / "transcriptomics.pkl"),
    "proteomics": pd.read_pickle(DATA_DIR / "proteomics.pkl"),
    "methylation": pd.read_pickle(DATA_DIR / "methylation.pkl"),
}

for name, df in omics.items():
    print(f"{name:15s}: {df.shape[0]:4d} patients x {df.shape[1]:6d} columns")
    assert "subtype" in df.columns, f"{name} is missing the subtype column"

# The samples are already aligned, so we can use the index from any table.
patient_ids = omics["transcriptomics"].index

# The target is the subtype column.
y_raw = omics["transcriptomics"]["subtype"].copy()

# Feature matrices: drop only the target column.
X_views = {
    name: df.drop(columns=["subtype"]).copy()
    for name, df in omics.items()
}

print("\nSubtype counts:")
display(y_raw.value_counts())


## 2. Create one shared train/test split

Both deep learning strategies use the same split.

This keeps the comparison fair:

1. Early integration neural network
2. Multi-modal encoder neural network


In [ ]:
train_ids, test_ids = train_test_split(
    patient_ids,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_raw,
)

print(f"Training patients: {len(train_ids)}")
print(f"Test patients:     {len(test_ids)}")


## 3. Encode labels and scale features

Neural networks train more reliably when input features are on a comparable scale.

This is not biological preprocessing or imputation.  
It is model fitting hygiene: scalers are fit on the training set only and then applied to the test set.


In [ ]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(y_raw.loc[train_ids])
y_test = label_encoder.transform(y_raw.loc[test_ids])

class_names = label_encoder.classes_
n_classes = len(class_names)

print("Classes:", list(class_names))

# Fit one scaler per omic view using training patients only.
scalers = {}

X_train_views = {}
X_test_views = {}

for name, X in X_views.items():
    scaler = StandardScaler()

    X_train_views[name] = scaler.fit_transform(X.loc[train_ids]).astype(np.float32)
    X_test_views[name] = scaler.transform(X.loc[test_ids]).astype(np.float32)

    scalers[name] = scaler

input_dims = {name: X_train.shape[1] for name, X_train in X_train_views.items()}
input_dims


## 4. Helper functions for PyTorch training and evaluation

These helpers keep the modelling cells short.

The training loop is intentionally simple:

- mini-batch gradient descent
- cross-entropy loss for multi-class subtype prediction
- Adam optimizer
- no cross-validation


In [ ]:
def make_loader(X, y, batch_size=32, shuffle=True):
    """Create a PyTorch DataLoader for a single matrix X."""
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)
    dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def train_classifier(model, train_loader, n_epochs=50, lr=1e-3, weight_decay=1e-4):
    """Train a classifier and return the loss history."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    history = []

    for epoch in range(n_epochs):
        model.train()
        running_loss = 0.0

        for batch in train_loader:
            optimizer.zero_grad()

            # Some models receive one input tensor; some receive a list of tensors.
            *features, target = batch
            features = [x.to(device) for x in features]
            target = target.to(device)

            logits = model(*features)
            loss = criterion(logits, target)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * target.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        history.append(epoch_loss)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1:03d} | loss = {epoch_loss:.4f}")

    return history


@torch.no_grad()
def predict(model, loader):
    """Return predicted class indices for a DataLoader."""
    model.eval()
    preds = []

    for batch in loader:
        *features, _ = batch
        features = [x.to(device) for x in features]
        logits = model(*features)
        preds.append(logits.argmax(dim=1).cpu().numpy())

    return np.concatenate(preds)


def evaluate_predictions(y_true, y_pred, title):
    """Print common classification metrics."""
    print(title)
    print("-" * len(title))
    print(f"Accuracy:          {accuracy_score(y_true, y_pred):.3f}")
    print(f"Balanced accuracy: {balanced_accuracy_score(y_true, y_pred):.3f}")
    print()
    print(classification_report(y_true, y_pred, target_names=class_names))


## 5. Baseline deep learning approach: early integration MLP

Early integration concatenates all omics into one large feature matrix before modelling.

This is simple and often effective, but it treats all features as one undifferentiated block.


In [ ]:
X_train_early = np.concatenate(
    [X_train_views[name] for name in X_train_views],
    axis=1,
)

X_test_early = np.concatenate(
    [X_test_views[name] for name in X_test_views],
    axis=1,
)

print("Early integration training matrix:", X_train_early.shape)
print("Early integration test matrix:    ", X_test_early.shape)


In [ ]:
class EarlyIntegrationMLP(nn.Module):
    """A simple neural network for concatenated multi-omic features."""

    def __init__(self, input_dim, hidden_dim=128, embedding_dim=32, n_classes=2, dropout=0.25):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embedding_dim),
            nn.ReLU(),
        )

        self.classifier = nn.Linear(embedding_dim, n_classes)

    def forward(self, x):
        embedding = self.encoder(x)
        return self.classifier(embedding)

    def embed(self, x):
        return self.encoder(x)


early_model = EarlyIntegrationMLP(
    input_dim=X_train_early.shape[1],
    hidden_dim=128,
    embedding_dim=32,
    n_classes=n_classes,
)

early_loader = make_loader(X_train_early, y_train, batch_size=32, shuffle=True)
early_test_loader = make_loader(X_test_early, y_test, batch_size=64, shuffle=False)

early_history = train_classifier(
    early_model,
    early_loader,
    n_epochs=50,
    lr=1e-3,
    weight_decay=1e-4,
)


In [ ]:
early_pred = predict(early_model, early_test_loader)
evaluate_predictions(y_test, early_pred, "Early integration MLP")


## 6. Multi-modal encoder network

A multi-modal encoder keeps the omics separated at the input.

Each omic has its own small encoder:

- transcriptomics encoder
- proteomics encoder
- methylation encoder

The learned omic-specific embeddings are then concatenated into a shared multi-omic representation used for subtype prediction.


In [ ]:
def make_multiview_loader(X_views_dict, y, batch_size=32, shuffle=True):
    """Create a DataLoader where each batch contains one tensor per omic view."""
    tensors = [
        torch.tensor(X_views_dict[name], dtype=torch.float32)
        for name in X_views_dict
    ]
    tensors.append(torch.tensor(y, dtype=torch.long))

    dataset = TensorDataset(*tensors)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


class MultiOmicEncoder(nn.Module):
    """One encoder per omic view, followed by a shared classifier."""

    def __init__(self, input_dims, view_embedding_dim=16, hidden_dim=64, n_classes=2, dropout=0.25):
        super().__init__()

        self.view_names = list(input_dims.keys())

        self.encoders = nn.ModuleDict({
            name: nn.Sequential(
                nn.Linear(dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, view_embedding_dim),
                nn.ReLU(),
            )
            for name, dim in input_dims.items()
        })

        joint_dim = view_embedding_dim * len(input_dims)

        self.classifier = nn.Sequential(
            nn.Linear(joint_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, *views):
        embeddings = []

        for name, x in zip(self.view_names, views):
            embeddings.append(self.encoders[name](x))

        joint_embedding = torch.cat(embeddings, dim=1)
        return self.classifier(joint_embedding)

    def embed(self, *views):
        embeddings = []

        for name, x in zip(self.view_names, views):
            embeddings.append(self.encoders[name](x))

        return torch.cat(embeddings, dim=1)


multi_loader = make_multiview_loader(X_train_views, y_train, batch_size=32, shuffle=True)
multi_test_loader = make_multiview_loader(X_test_views, y_test, batch_size=64, shuffle=False)

multi_model = MultiOmicEncoder(
    input_dims=input_dims,
    view_embedding_dim=16,
    hidden_dim=64,
    n_classes=n_classes,
)

multi_history = train_classifier(
    multi_model,
    multi_loader,
    n_epochs=50,
    lr=1e-3,
    weight_decay=1e-4,
)


In [ ]:
multi_pred = predict(multi_model, multi_test_loader)
evaluate_predictions(y_test, multi_pred, "Multi-modal encoder network")


## 7. Compare model performance

Because both models used the same train/test split, the comparison is direct.


In [ ]:
results = pd.DataFrame({
    "model": ["Early integration MLP", "Multi-modal encoder"],
    "accuracy": [
        accuracy_score(y_test, early_pred),
        accuracy_score(y_test, multi_pred),
    ],
    "balanced_accuracy": [
        balanced_accuracy_score(y_test, early_pred),
        balanced_accuracy_score(y_test, multi_pred),
    ],
})

display(results.sort_values("balanced_accuracy", ascending=False))


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(early_history, label="Early integration MLP")
plt.plot(multi_history, label="Multi-modal encoder")
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Training curves")
plt.legend()
plt.show()


## 8. Inspect the learned multi-omic embedding space

The multi-modal encoder produces one compact vector per patient.

This embedding is a learned multi-omic profile.  
We can visualize it using PCA to see whether subtype structure appears in the learned representation.


In [ ]:
@torch.no_grad()
def get_embeddings(model, loader):
    """Extract embeddings from a model that implements an embed method."""
    model.eval()
    embeddings = []
    labels = []

    for batch in loader:
        *features, target = batch
        features = [x.to(device) for x in features]
        emb = model.embed(*features)
        embeddings.append(emb.cpu().numpy())
        labels.append(target.numpy())

    return np.vstack(embeddings), np.concatenate(labels)


train_embedding, train_embedding_y = get_embeddings(multi_model, multi_loader)
test_embedding, test_embedding_y = get_embeddings(multi_model, multi_test_loader)

print("Train embedding shape:", train_embedding.shape)
print("Test embedding shape: ", test_embedding.shape)


In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)

embedding_2d = pca.fit_transform(test_embedding)

embedding_df = pd.DataFrame({
    "PC1": embedding_2d[:, 0],
    "PC2": embedding_2d[:, 1],
    "subtype": label_encoder.inverse_transform(test_embedding_y),
}, index=test_ids)

display(embedding_df.head())


In [ ]:
plt.figure(figsize=(6, 5))

for subtype in class_names:
    subset = embedding_df[embedding_df["subtype"] == subtype]
    plt.scatter(subset["PC1"], subset["PC2"], label=subtype, alpha=0.8)

plt.xlabel("Embedding PC1")
plt.ylabel("Embedding PC2")
plt.title("Multi-modal encoder embedding space")
plt.legend()
plt.show()


## 9. Takeaways

In this notebook we compared two neural network approaches for multi-omic subtype prediction.

### Early integration MLP

- Concatenates all features before modelling.
- Simple and useful as a baseline.
- Does not explicitly represent omic-specific structure.

### Multi-modal encoder

- Keeps each omic separate at the input.
- Learns omic-specific embeddings before combining them.
- Produces a compact patient-level embedding that can be reused downstream.

## Link to the next part

The key output from this section is the **multi-omic embedding matrix**.

In the next part, this representation can be used for:

- patient stratification
- clustering
- visualisation
- survival modelling
- biological interpretation of latent patient profiles
